In [1]:
!nvidia-smi

Sun Aug 30 03:24:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import files

uploaded = files.upload()

Saving distilbert_phishing_final_project.zip to distilbert_phishing_final_project.zip


In [3]:
!unzip -q distilbert_phishing_final_project.zip

In [4]:
!ls distilbert_phishing_final

config.py	      README.md			    train_distilbert.py
evaluate_external.py  requirements.txt
inference.py	      train_distilbert_colab.ipynb


In [5]:
%cd distilbert_phishing_final

/content/distilbert_phishing_final


In [6]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [7]:
import torch
import transformers
import datasets
import accelerate
import sklearn
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.0.0
Accelerate: 1.14.0
Hugging Face Hub: 0.36.2
CUDA available: True
GPU: Tesla T4


In [8]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset")

print(dataset)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv:   0%|          | 0.00/52.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Email Text', 'Email Type'],
        num_rows: 18650
    })
})


In [9]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (18650, 3)

Columns:
['Unnamed: 0', 'Email Text', 'Email Type']

First 5 rows:


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [10]:
print(df["Email Type"].value_counts())

Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64


In [11]:
print(df.isnull().sum())

Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64


In [12]:
import pandas as pd
import re

# Keep only the columns we actually need
df = df[["Email Text", "Email Type"]].copy()

# Remove emails with no text
df = df.dropna(subset=["Email Text"])

# Convert email text to string
df["Email Text"] = df["Email Text"].astype(str)

# Clean excessive whitespace
df["Email Text"] = df["Email Text"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

# Remove emails that became empty
df = df[df["Email Text"].str.len() > 0]

# Convert labels to numbers
df["label"] = df["Email Type"].map({
    "Safe Email": 0,
    "Phishing Email": 1
})

# Remove anything that didn't map correctly
df = df.dropna(subset=["label"])

# Convert labels to integer
df["label"] = df["label"].astype(int)

print("Shape after cleaning:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Shape after cleaning: (18631, 3)

Label distribution:
label
0    11322
1     7309
Name: count, dtype: int64

Missing values:
Email Text    0
Email Type    0
label         0
dtype: int64


In [13]:
print("Total emails:", len(df))
print("Duplicate emails:", df["Email Text"].duplicated().sum())

Total emails: 18631
Duplicate emails: 1117


In [14]:
df = df.drop_duplicates(subset=["Email Text"]).reset_index(drop=True)

print("After removing duplicates:", len(df))

After removing duplicates: 17514


In [15]:
print(df.head())

print("\nFinal shape:", df.shape)

print("\nFinal label distribution:")
print(df["label"].value_counts())

print("\nFinal percentages:")
print(df["label"].value_counts(normalize=True) * 100)

                                          Email Text      Email Type  label
0  re : 6 . 1100 , disc : uniformitarianism , re ...      Safe Email      0
1  the other side of * galicismos * * galicismo *...      Safe Email      0
2  re : equistar deal tickets are you still avail...      Safe Email      0
3  Hello I am your hot lil horny toy. I am the on...  Phishing Email      1
4  software at incredibly low prices ( 86 % lower...  Phishing Email      1

Final shape: (17514, 3)

Final label distribution:
label
0    10977
1     6537
Name: count, dtype: int64

Final percentages:
label
0    62.675574
1    37.324426
Name: proportion, dtype: float64


In [16]:
from sklearn.model_selection import train_test_split

# First split: 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

# Second split: divide the remaining 20% equally
# → 10% validation + 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

TRAIN: 14011
VALIDATION: 1751
TEST: 1752


In [17]:
print("\nTRAIN distribution:")
print(train_df["label"].value_counts())

print("\nVALIDATION distribution:")
print(val_df["label"].value_counts())

print("\nTEST distribution:")
print(test_df["label"].value_counts())


TRAIN distribution:
label
0    8781
1    5230
Name: count, dtype: int64

VALIDATION distribution:
label
0    1098
1     653
Name: count, dtype: int64

TEST distribution:
label
0    1098
1     654
Name: count, dtype: int64


In [18]:
train_texts = set(train_df["Email Text"])
val_texts = set(val_df["Email Text"])
test_texts = set(test_df["Email Text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "LEGITIMATE",
        1: "PHISHING"
    },
    label2id={
        "LEGITIMATE": 0,
        "PHISHING": 1
    }
)

print("Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully!


In [20]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [21]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

Total parameters: 66,955,010
Trainable parameters: 66,955,010


In [22]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Using device:", device)

Using device: cuda


In [23]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["Email Text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["Email Text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["Email Text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['Email Text', 'label'],
    num_rows: 14011
})
Dataset({
    features: ['Email Text', 'label'],
    num_rows: 1751
})
Dataset({
    features: ['Email Text', 'label'],
    num_rows: 1752
})


In [24]:
MAX_LENGTH = 512

def tokenize_function(batch):
    return tokenizer(
        batch["Email Text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Email Text"]
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Email Text"]
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["Email Text"]
)

print(train_tokenized)

Map:   0%|          | 0/14011 [00:00<?, ? examples/s]

Map:   0%|          | 0/1751 [00:00<?, ? examples/s]

Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 14011
})


In [25]:
print(train_tokenized[0])

{'label': 0, 'input_ids': [101, 20604, 20231, 1998, 19476, 2655, 2005, 4981, 3034, 1024, 1996, 20231, 1997, 20604, 4155, 1006, 2238, 2656, 1011, 2676, 1007, 8395, 1024, 1996, 19476, 1997, 20604, 4155, 1006, 2238, 2603, 1007, 2044, 1037, 2200, 3144, 2034, 20604, 20231, 3034, 2012, 15529, 2378, 2089, 2687, 1010, 1996, 2533, 1997, 15397, 2012, 1996, 2118, 1997, 4307, 2012, 27929, 1011, 28843, 18675, 29474, 2005, 1996, 2117, 20604, 20231, 3034, 2000, 2022, 2218, 2238, 2656, 1011, 2676, 1010, 2639, 1012, 1996, 3034, 2097, 2022, 11677, 2011, 1037, 8395, 2239, 20604, 19476, 2000, 2022, 2218, 2238, 2603, 1010, 2639, 1012, 2119, 2824, 2097, 2202, 2173, 2076, 1996, 12158, 2554, 1997, 2637, 2621, 2820, 1006, 2238, 17465, 1011, 2251, 2382, 1007, 2029, 2097, 2022, 2218, 2012, 1996, 2118, 1997, 4307, 2012, 27929, 1011, 28843, 1006, 4037, 1024, 8299, 1024, 1013, 1013, 7479, 1012, 10272, 2386, 1012, 21318, 14194, 1012, 3968, 2226, 1013, 17002, 7076, 2102, 1007, 1012, 6818, 2097, 2022, 3479, 2006, 1996

In [26]:
lengths = [
    len(x["input_ids"])
    for x in train_tokenized.select(range(min(1000, len(train_tokenized))))
]

print("Shortest:", min(lengths))
print("Longest:", max(lengths))
print("Average:", sum(lengths) / len(lengths))

Shortest: 3
Longest: 512
Average: 274.738


In [27]:
MAX_LENGTH = 512
STRIDE = 128

def tokenize_with_chunks(batch):
    return tokenizer(
        batch["Email Text"],
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        padding=False
    )

In [28]:
# Find the longest email based on token count
token_lengths = []

for text in df["Email Text"]:
    tokens = tokenizer(
        text,
        truncation=False,
        add_special_tokens=True
    )["input_ids"]
    token_lengths.append(len(tokens))

df["token_length"] = token_lengths

print("Longest email:", df["token_length"].max(), "tokens")

long_email = df.loc[df["token_length"].idxmax(), "Email Text"]

print("\nCharacters:", len(long_email))
print("First 500 characters:\n")
print(long_email[:500])

Token indices sequence length is longer than the specified maximum sequence length for this model (616 > 512). Running this sequence through the model will result in indexing errors


Longest email: 4132149 tokens

Characters: 17036692
First 500 characters:

0,"Subject: great part-time or summer job ! * * * * * * * * * * * * * * * we have display boxes with credit applications that we need to place in the small owner-operated stores in your area . here is what you do : 1 . introduce yourself to the store owner or manager . 2 . use our 90 % effective script which tells them how this little display box will save their customers hundreds of dollars , be a drawing card for their business , and make them from $ 5 . 00 to $ 15 . 00 or more for every app s


In [29]:
long_email_chunks = tokenizer(
    long_email,
    truncation=True,
    max_length=MAX_LENGTH,
    stride=STRIDE,
    return_overflowing_tokens=True,
    padding=False
)

print("Original token count:",
      len(tokenizer(long_email, truncation=False)["input_ids"]))

print("Number of chunks:",
      len(long_email_chunks["input_ids"]))

print("\nChunk lengths:")

for i, chunk in enumerate(long_email_chunks["input_ids"]):
    print(f"Chunk {i + 1}: {len(chunk)} tokens")

Original token count: 4132149
Number of chunks: 10817

Chunk lengths:
Chunk 1: 512 tokens
Chunk 2: 512 tokens
Chunk 3: 512 tokens
Chunk 4: 512 tokens
Chunk 5: 512 tokens
Chunk 6: 512 tokens
Chunk 7: 512 tokens
Chunk 8: 512 tokens
Chunk 9: 512 tokens
Chunk 10: 512 tokens
Chunk 11: 512 tokens
Chunk 12: 512 tokens
Chunk 13: 512 tokens
Chunk 14: 512 tokens
Chunk 15: 512 tokens
Chunk 16: 512 tokens
Chunk 17: 512 tokens
Chunk 18: 512 tokens
Chunk 19: 512 tokens
Chunk 20: 512 tokens
Chunk 21: 512 tokens
Chunk 22: 512 tokens
Chunk 23: 512 tokens
Chunk 24: 512 tokens
Chunk 25: 512 tokens
Chunk 26: 512 tokens
Chunk 27: 512 tokens
Chunk 28: 512 tokens
Chunk 29: 512 tokens
Chunk 30: 512 tokens
Chunk 31: 512 tokens
Chunk 32: 512 tokens
Chunk 33: 512 tokens
Chunk 34: 512 tokens
Chunk 35: 512 tokens
Chunk 36: 512 tokens
Chunk 37: 512 tokens
Chunk 38: 512 tokens
Chunk 39: 512 tokens
Chunk 40: 512 tokens
Chunk 41: 512 tokens
Chunk 42: 512 tokens
Chunk 43: 512 tokens
Chunk 44: 512 tokens
Chunk 45: 512 t

In [30]:
from transformers import AutoTokenizer

# Load the tokenizer for our existing DistilBERT model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

MAX_LENGTH = 512
CHUNK_SIZE = 480
OVERLAP = 50

def chunk_email(text):
    """
    Split an email into overlapping chunks that DistilBERT can process.
    """

    # Convert email into tokens
    tokens = tokenizer.encode(
        str(text),
        add_special_tokens=False
    )

    chunks = []

    # Split into chunks
    start = 0

    while start < len(tokens):
        end = start + CHUNK_SIZE
        chunk_tokens = tokens[start:end]

        # Convert tokens back to text
        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk_text)

        # Move forward while keeping overlap
        start += CHUNK_SIZE - OVERLAP

    return chunks

In [32]:
test_email = df["Email Text"].iloc[0]

chunks = chunk_email(test_email)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    token_count = len(
        tokenizer.encode(
            chunk,
            add_special_tokens=True
        )
    )
    print(f"Chunk {i+1}: {token_count} tokens")

Number of chunks: 1
Chunk 1: 254 tokens


In [33]:
long_email = df.loc[
    df["Email Text"].map(
        lambda x: len(tokenizer.encode(
            str(x),
            add_special_tokens=False
        ))
    ).idxmax(),
    "Email Text"
]

print("Characters:", len(long_email))

long_chunks = chunk_email(long_email)

print("Number of chunks:", len(long_chunks))

print("\nFirst 5 chunks:")

for i, chunk in enumerate(long_chunks[:5]):
    token_count = len(
        tokenizer.encode(
            chunk,
            add_special_tokens=True
        )
    )
    print(f"Chunk {i+1}: {token_count} tokens")

Token indices sequence length is longer than the specified maximum sequence length for this model (614 > 512). Running this sequence through the model will result in indexing errors


Characters: 17036692
Number of chunks: 9610

First 5 chunks:
Chunk 1: 482 tokens
Chunk 2: 482 tokens
Chunk 3: 482 tokens
Chunk 4: 482 tokens
Chunk 5: 482 tokens


In [34]:
# ============================================================
# STEP 9: Convert our Pandas dataframes into Hugging Face
# Dataset objects
# ============================================================

from datasets import Dataset

# Keep ONLY the two columns required for training:
#   Email Text -> the input to DistilBERT
#   label      -> 0 = legitimate, 1 = phishing
train_dataset = Dataset.from_pandas(
    train_df[["Email Text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["Email Text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["Email Text", "label"]],
    preserve_index=False
)

print("Training dataset:")
print(train_dataset)

print("\nValidation dataset:")
print(val_dataset)

print("\nTest dataset:")
print(test_dataset)

Training dataset:
Dataset({
    features: ['Email Text', 'label'],
    num_rows: 14011
})

Validation dataset:
Dataset({
    features: ['Email Text', 'label'],
    num_rows: 1751
})

Test dataset:
Dataset({
    features: ['Email Text', 'label'],
    num_rows: 1752
})


In [35]:
# ============================================================
# STEP 10: Tokenization
# ============================================================

MAX_LENGTH = 512

def tokenize_emails(batch):
    """
    Convert email text into DistilBERT input tokens.

    truncation=True:
        Ensures no input passed to the model is longer than
        DistilBERT's 512-token limit.

    max_length=512:
        Maximum sequence length supported by our model.
    """

    return tokenizer(
        batch["Email Text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


# Tokenize the training set
train_tokenized = train_dataset.map(
    tokenize_emails,
    batched=True,
    remove_columns=["Email Text"]
)

# Tokenize the validation set
val_tokenized = val_dataset.map(
    tokenize_emails,
    batched=True,
    remove_columns=["Email Text"]
)

# Tokenize the test set
test_tokenized = test_dataset.map(
    tokenize_emails,
    batched=True,
    remove_columns=["Email Text"]
)

print("Tokenization complete!")
print(train_tokenized)

Map:   0%|          | 0/14011 [00:00<?, ? examples/s]

Map:   0%|          | 0/1751 [00:00<?, ? examples/s]

Map:   0%|          | 0/1752 [00:00<?, ? examples/s]

Tokenization complete!
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 14011
})


In [36]:
# ============================================================
# STEP 11: Verify tokenization
# ============================================================

print("First training example:")
print(train_tokenized[0])

print("\nNumber of input tokens:",
      len(train_tokenized[0]["input_ids"]))

print("\nLabel:",
      train_tokenized[0]["label"])

First training example:
{'label': 0, 'input_ids': [101, 20604, 20231, 1998, 19476, 2655, 2005, 4981, 3034, 1024, 1996, 20231, 1997, 20604, 4155, 1006, 2238, 2656, 1011, 2676, 1007, 8395, 1024, 1996, 19476, 1997, 20604, 4155, 1006, 2238, 2603, 1007, 2044, 1037, 2200, 3144, 2034, 20604, 20231, 3034, 2012, 15529, 2378, 2089, 2687, 1010, 1996, 2533, 1997, 15397, 2012, 1996, 2118, 1997, 4307, 2012, 27929, 1011, 28843, 18675, 29474, 2005, 1996, 2117, 20604, 20231, 3034, 2000, 2022, 2218, 2238, 2656, 1011, 2676, 1010, 2639, 1012, 1996, 3034, 2097, 2022, 11677, 2011, 1037, 8395, 2239, 20604, 19476, 2000, 2022, 2218, 2238, 2603, 1010, 2639, 1012, 2119, 2824, 2097, 2202, 2173, 2076, 1996, 12158, 2554, 1997, 2637, 2621, 2820, 1006, 2238, 17465, 1011, 2251, 2382, 1007, 2029, 2097, 2022, 2218, 2012, 1996, 2118, 1997, 4307, 2012, 27929, 1011, 28843, 1006, 4037, 1024, 8299, 1024, 1013, 1013, 7479, 1012, 10272, 2386, 1012, 21318, 14194, 1012, 3968, 2226, 1013, 17002, 7076, 2102, 1007, 1012, 6818, 2097

In [37]:
# Check that no tokenized training example exceeds 512 tokens

max_length_found = max(
    len(example["input_ids"])
    for example in train_tokenized
)

print("Maximum tokenized length:", max_length_found)

Maximum tokenized length: 512


In [38]:
# ============================================================
# STEP 12: Data Collator
# ============================================================

from transformers import DataCollatorWithPadding

# Dynamically pads emails within each batch.
#
# This is more memory-efficient than padding every email
# to 512 tokens before training.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator ready.")

Data collator ready.


In [39]:
# ============================================================
# STEP 13: Move DistilBERT to the GPU
# ============================================================

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [40]:
# ============================================================
# STEP 14: Configure DistilBERT Training
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)


# ------------------------------------------------------------
# 1. Evaluation metrics
# ------------------------------------------------------------
# We calculate more than accuracy because phishing detection
# needs to consider false negatives and false positives.
#
# label:
#   0 = legitimate
#   1 = phishing
#
# The model produces logits for both classes.
# We convert them into probabilities using softmax.
# ------------------------------------------------------------

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits → probabilities
    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=-1
    ).numpy()

    # Probability of phishing
    phishing_probability = probabilities[:, 1]

    # Default classification threshold
    predictions = (
        phishing_probability >= 0.50
    ).astype(int)

    # Calculate metrics
    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="binary",
            zero_division=0
        )
    )

    roc_auc = roc_auc_score(
        labels,
        phishing_probability
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }


# ------------------------------------------------------------
# 2. Training configuration
# ------------------------------------------------------------

training_args = TrainingArguments(

    # Where checkpoints will temporarily be stored
    output_dir="./distilbert_checkpoints",

    # Train for 3 complete passes through the training data
    num_train_epochs=3,

    # Standard fine-tuning learning rate for BERT-family models
    learning_rate=2e-5,

    # Helps prevent excessive overfitting
    weight_decay=0.01,

    # Number of examples processed at once
    per_device_train_batch_size=16,

    # Validation batch size can be larger because
    # gradients aren't calculated during evaluation
    per_device_eval_batch_size=32,

    # Gradually increase learning rate at the beginning
    warmup_ratio=0.1,

    # Evaluate after every epoch
    eval_strategy="epoch",

    # Save a checkpoint after every epoch
    save_strategy="epoch",

    # Keep the checkpoint with the best validation F1
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Show training progress
    logging_steps=50,

    # Don't send anything to external experiment trackers
    report_to="none",

    # Reproducibility
    seed=42,

    # Use T4 mixed precision to reduce GPU memory usage
    fp16=torch.cuda.is_available()
)

print("Training configuration created successfully.")
print(training_args)

Training configuration created successfully.
TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=Interval

In [41]:
# ============================================================
# STEP 15: Create the Hugging Face Trainer
# ============================================================

trainer = Trainer(

    # Our DistilBERT model
    model=model,

    # Training configuration from Step 14
    args=training_args,

    # Training data
    train_dataset=train_tokenized,

    # Validation data
    eval_dataset=val_tokenized,

    # Tokenizer
    processing_class=tokenizer,

    # Dynamic padding
    data_collator=data_collator,

    # Accuracy / Precision / Recall / F1 / ROC-AUC
    compute_metrics=compute_metrics,

    # Stop if validation F1 stops improving
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("Trainer created successfully.")

Trainer created successfully.


In [42]:
# ============================================================
# STEP 16: Final pre-training check
# ============================================================

print("Model device:", next(model.parameters()).device)
print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))
print("Test examples:", len(test_tokenized))
print("Batch size:", training_args.per_device_train_batch_size)
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)

Model device: cuda:0
Training examples: 14011
Validation examples: 1751
Test examples: 1752
Batch size: 16
Epochs: 3
Learning rate: 2e-05


In [43]:
# ============================================================
# STEP 17: TRAIN THE FINAL DISTILBERT MODEL
# ============================================================

print("Starting DistilBERT training...")
print("GPU:", torch.cuda.get_device_name(0))
print()

train_result = trainer.train()

print("\n================================================")
print("TRAINING COMPLETE")
print("================================================")

print("\nTraining statistics:")
print(train_result)

Starting DistilBERT training...
GPU: Tesla T4



Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.063600,0.062063,0.980011,0.968182,0.978560,0.973343,0.998801
2,0.028100,0.068501,0.982296,0.979938,0.972435,0.976172,0.998992
3,0.008200,0.077665,0.983438,0.982972,0.972435,0.977675,0.999124



TRAINING COMPLETE

Training statistics:
TrainOutput(global_step=2628, training_loss=0.06639969708962165, metrics={'train_runtime': 622.3952, 'train_samples_per_second': 67.534, 'train_steps_per_second': 4.222, 'total_flos': 5559176527272192.0, 'train_loss': 0.06639969708962165, 'epoch': 3.0})


In [44]:
# ============================================================
# STEP 18: FINAL TEST EVALUATION
# ============================================================

print("Evaluating the final DistilBERT model on the")
print("completely untouched TEST dataset...\n")

test_results = trainer.evaluate(
    eval_dataset=test_tokenized
)

print("\n================================================")
print("FINAL TEST RESULTS")
print("================================================")

for metric, value in test_results.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

Evaluating the final DistilBERT model on the
completely untouched TEST dataset...




FINAL TEST RESULTS
eval_loss: 0.0347
eval_accuracy: 0.9932
eval_precision: 0.9938
eval_recall: 0.9878
eval_f1: 0.9908
eval_roc_auc: 0.9995
eval_runtime: 10.3364
eval_samples_per_second: 169.4980
eval_steps_per_second: 5.3210
epoch: 3.0000


In [45]:
# ============================================================
# STEP 19: LOAD EXTERNAL DATASET
# ============================================================

from datasets import load_dataset

external_dataset = load_dataset(
    "jmmrcp/Phishing-IBM-Dataset"
)

print(external_dataset)

README.md:   0%|          | 0.00/470 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.98M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13337 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3335 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['index', 'message', 'Classify'],
        num_rows: 13337
    })
    test: Dataset({
        features: ['index', 'message', 'Classify'],
        num_rows: 3335
    })
})


In [46]:
# Show the columns in the external dataset

for split_name, split_data in external_dataset.items():
    print("\nSplit:", split_name)
    print("Columns:", split_data.column_names)
    print("Number of examples:", len(split_data))
    print("\nFirst example:")
    print(split_data[0])


Split: train
Columns: ['index', 'message', 'Classify']
Number of examples: 13337

First example:
{'index': 8213, 'message': 'expense ground rules what level of expense do you want to be notified about ? ( e . g . i want to authorise kal to spend about $ 5 , 000 over the next couple of weeks on preliminary work for marketing materials to support a relaunch of enrononline ) dave', 'Classify': 'Normal'}

Split: test
Columns: ['index', 'message', 'Classify']
Number of examples: 3335

First example:
{'index': 17671, 'message': "meds ? we have it all right here effort make heads turn by wearing one of this season ' s most coveted wholesale repliica bags , watches or whatever luxury ! start shopping today and find out why so many other women are choosing to carry our products year round . http : / / aaqa . starsleekgreat . com / li / addidas , bally , bvlgari , burberry , cartier , chanel , christian dior , dunhill , dupont , escada , fendi , ferragamo , gucci , hermes , iwc , jacob & co . ,

In [47]:
# ============================================================
# STEP 19.1: INSPECT EXTERNAL DATASET LABELS
# ============================================================

# Get the external test set
external_test = external_dataset["test"]

# Show all unique label values
print("Unique labels:")
print(set(external_test["Classify"]))

# Show the distribution
print("\nLabel distribution:")
from collections import Counter

print(Counter(external_test["Classify"]))

# Show a few examples
print("\nFirst 5 examples:")

for i in range(5):
    print("\nExample", i + 1)
    print("Label :", external_test[i]["Classify"])
    print("Text  :", external_test[i]["message"][:300])

Unique labels:
{'Normal', 'Phishing'}

Label distribution:
Counter({'Normal': 1686, 'Phishing': 1649})

First 5 examples:

Example 1
Label : Phishing
Text  : meds ? we have it all right here effort make heads turn by wearing one of this season ' s most coveted wholesale repliica bags , watches or whatever luxury ! start shopping today and find out why so many other women are choosing to carry our products year round . http : / / aaqa . starsleekgreat . c

Example 2
Label : Normal
Text  : hpl nom for october 6 , 2000 ( see attached file : hpll 006 . xls ) - hpll 006 . xls

Example 3
Label : Phishing
Text  : re : new page hi sweetie , come see the most beautiful , sweet . . . - - > 18 year old girls bare it all ! < - - http : / / freexmovies . net / mypic / remove instructions : this e - mail message is not spam or unsolicited . this e - mail address has joined or requested information in the past . if 

Example 4
Label : Normal
Text  : re : budget and a lemmons jr . , billy subject : re

In [48]:
# ============================================================
# STEP 20: EXTERNAL DATASET EVALUATION
# ============================================================
#
# IMPORTANT:
# We are NOT training on this dataset.
#
# This is an independent test of our already-trained
# DistilBERT model.
# ============================================================

import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Get the external TEST dataset
# ------------------------------------------------------------

external_test = external_dataset["test"]

# Extract email text
external_texts = external_test["message"]

# Convert labels:
# Normal   = 0
# Phishing = 1

external_labels = np.array([
    1 if label == "Phishing" else 0
    for label in external_test["Classify"]
])

print("External test emails:", len(external_texts))
print("Normal:", np.sum(external_labels == 0))
print("Phishing:", np.sum(external_labels == 1))


# ------------------------------------------------------------
# 2. Put model in evaluation mode
# ------------------------------------------------------------

model.eval()

# Make sure model is on the GPU
model.to(device)


# ------------------------------------------------------------
# 3. Run predictions in batches
# ------------------------------------------------------------
#
# We use batches instead of processing all 3,335 emails
# simultaneously, which keeps GPU memory under control.
# ------------------------------------------------------------

BATCH_SIZE = 32

all_phishing_probabilities = []

for start in range(0, len(external_texts), BATCH_SIZE):

    # Select one batch
    batch_texts = external_texts[
        start:start + BATCH_SIZE
    ]

    # Tokenize
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    # Move tensors to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # No gradients needed during evaluation
    with torch.no_grad():

        outputs = model(**inputs)

        # Convert logits → probabilities
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        # Probability of PHISHING
        phishing_probs = probabilities[:, 1]

        all_phishing_probabilities.extend(
            phishing_probs.cpu().numpy()
        )


# Convert to NumPy array
all_phishing_probabilities = np.array(
    all_phishing_probabilities
)


# ------------------------------------------------------------
# 4. Convert probabilities into predictions
# ------------------------------------------------------------

THRESHOLD = 0.50

external_predictions = (
    all_phishing_probabilities >= THRESHOLD
).astype(int)


# ------------------------------------------------------------
# 5. Calculate metrics
# ------------------------------------------------------------

accuracy = accuracy_score(
    external_labels,
    external_predictions
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        external_labels,
        external_predictions,
        average="binary",
        zero_division=0
    )
)

roc_auc = roc_auc_score(
    external_labels,
    all_phishing_probabilities
)

cm = confusion_matrix(
    external_labels,
    external_predictions
)


# ------------------------------------------------------------
# 6. Display final external results
# ------------------------------------------------------------

print("\n")
print("================================================")
print("EXTERNAL DATASET RESULTS")
print("================================================")

print(f"Accuracy :  {accuracy:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall   :  {recall:.4f}")
print(f"F1 Score :  {f1:.4f}")
print(f"ROC-AUC  :  {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\n================================================")

External test emails: 3335
Normal: 1686
Phishing: 1649


EXTERNAL DATASET RESULTS
Accuracy :  0.9901
Precision:  0.9939
Recall   :  0.9861
F1 Score :  0.9900
ROC-AUC  :  0.9994

Confusion Matrix:
[[1676   10]
 [  23 1626]]



In [49]:
# ============================================================
# STEP 21: SAVE THE FINAL DISTILBERT MODEL
# ============================================================

import os
import json

# Final model directory
FINAL_MODEL_DIR = "./distilbert_model_final"

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ------------------------------------------------------------
# Save the trained DistilBERT model
# ------------------------------------------------------------

trainer.save_model(FINAL_MODEL_DIR)

# ------------------------------------------------------------
# Save the tokenizer
# ------------------------------------------------------------

tokenizer.save_pretrained(FINAL_MODEL_DIR)

# ------------------------------------------------------------
# Save our label mapping
# ------------------------------------------------------------

label_mapping = {
    "0": "legitimate",
    "1": "phishing"
}

with open(
    f"{FINAL_MODEL_DIR}/label_mapping.json",
    "w"
) as f:
    json.dump(label_mapping, f, indent=4)

print("FINAL MODEL SAVED")
print("------------------")

print(os.listdir(FINAL_MODEL_DIR))

FINAL MODEL SAVED
------------------
['tokenizer_config.json', 'training_args.bin', 'tokenizer.json', 'config.json', 'label_mapping.json', 'vocab.txt', 'special_tokens_map.json', 'model.safetensors']


In [50]:
# ============================================================
# CyberMail Intelligence
# DistilBERT Phishing Email Classifier
# ============================================================
#
# This module loads the FINAL trained DistilBERT model and
# provides one simple function:
#
#     classifier.predict(email_body)
#
# The backend will eventually send the COMPLETE email body
# to this class.
#
# The class handles:
#   1. Short emails
#   2. Long emails using overlapping chunks
#   3. Chunk-level predictions
#   4. Email-level probability aggregation
#
# ============================================================

import json
import os

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


class DistilBERTPhishingClassifier:

    def __init__(
        self,
        model_dir="distilbert_model_final",
        max_length=512,
        chunk_size=480,
        overlap=50
    ):
        """
        Load the FINAL DistilBERT model.

        Parameters
        ----------
        model_dir : str
            Folder containing model.safetensors and tokenizer files.

        max_length : int
            Maximum sequence length supported by DistilBERT.

        chunk_size : int
            Number of actual email tokens in each chunk.

        overlap : int
            Number of tokens shared between consecutive chunks.
        """

        # ----------------------------------------------------
        # Basic configuration
        # ----------------------------------------------------

        self.model_dir = model_dir
        self.max_length = max_length
        self.chunk_size = chunk_size
        self.overlap = overlap

        # ----------------------------------------------------
        # Select GPU if available
        # ----------------------------------------------------

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # ----------------------------------------------------
        # Load tokenizer
        # ----------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_dir
        )

        # ----------------------------------------------------
        # Load FINAL trained model
        # ----------------------------------------------------

        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_dir
        )

        # Move model to GPU / CPU
        self.model.to(self.device)

        # Evaluation mode
        self.model.eval()

        # ----------------------------------------------------
        # Load label mapping
        # ----------------------------------------------------

        mapping_path = os.path.join(
            self.model_dir,
            "label_mapping.json"
        )

        if os.path.exists(mapping_path):

            with open(mapping_path, "r") as f:
                self.label_mapping = json.load(f)

        else:

            # Fallback mapping
            self.label_mapping = {
                "0": "legitimate",
                "1": "phishing"
            }

        print("DistilBERT loaded successfully.")
        print("Device:", self.device)

    # ========================================================
    # TOKENIZE EMAIL INTO OVERLAPPING CHUNKS
    # ========================================================

    def _create_chunks(self, email_body):
        """
        Convert a potentially long email into overlapping chunks.

        Every chunk stays below DistilBERT's 512-token limit.
        """

        # Convert text → token IDs
        tokens = self.tokenizer.encode(
            email_body,
            add_special_tokens=False
        )

        # Short email:
        # no chunking required
        if len(tokens) <= self.chunk_size:

            return [email_body]

        chunks = []

        # How far we move forward after every chunk
        step = self.chunk_size - self.overlap

        start = 0

        while start < len(tokens):

            # Select this chunk
            chunk_tokens = tokens[
                start:start + self.chunk_size
            ]

            # Convert tokens back into text
            chunk_text = self.tokenizer.decode(
                chunk_tokens,
                skip_special_tokens=True
            )

            chunks.append(chunk_text)

            # Move forward while retaining overlap
            start += step

        return chunks

    # ========================================================
    # PREDICT ONE CHUNK
    # ========================================================

    @torch.inference_mode()
    def _predict_chunk(self, chunk):
        """
        Predict phishing probability for one chunk.
        """

        # Tokenize the chunk
        inputs = self.tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length
        )

        # Move tensors to same device as model
        inputs = {
            key: value.to(self.device)
            for key, value in inputs.items()
        }

        # Run DistilBERT
        outputs = self.model(**inputs)

        # Convert logits → probabilities
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

        # Class 0 = legitimate
        # Class 1 = phishing

        legitimate_probability = float(
            probabilities[0].cpu()
        )

        phishing_probability = float(
            probabilities[1].cpu()
        )

        return {
            "legitimate_probability": legitimate_probability,
            "phishing_probability": phishing_probability
        }

    # ========================================================
    # PREDICT COMPLETE EMAIL
    # ========================================================

    def predict(self, email_body):
        """
        Analyze a complete email body.

        Returns ONE email-level result.
        """

        # ----------------------------------------------------
        # Basic input validation
        # ----------------------------------------------------

        if email_body is None:

            raise ValueError(
                "email_body cannot be None"
            )

        email_body = str(email_body).strip()

        if not email_body:

            raise ValueError(
                "email_body cannot be empty"
            )

        # ----------------------------------------------------
        # Create chunks
        # ----------------------------------------------------

        chunks = self._create_chunks(
            email_body
        )

        # ----------------------------------------------------
        # Predict each chunk
        # ----------------------------------------------------

        chunk_results = []

        for chunk in chunks:

            result = self._predict_chunk(
                chunk
            )

            chunk_results.append(result)

        # ----------------------------------------------------
        # Aggregate chunk probabilities
        # ----------------------------------------------------
        #
        # We use MAX here deliberately.
        #
        # If one section of a long email strongly indicates
        # phishing, we don't want several harmless sections
        # to dilute that signal.
        #
        # Example:
        #
        # Chunk 1 → 0.04
        # Chunk 2 → 0.11
        # Chunk 3 → 0.96  ← suspicious section
        # Chunk 4 → 0.08
        #
        # Email probability → 0.96
        #
        # ----------------------------------------------------

        phishing_probability = max(
            result["phishing_probability"]
            for result in chunk_results
        )

        legitimate_probability = 1.0 - phishing_probability

        # ----------------------------------------------------
        # Final classification
        # ----------------------------------------------------

        threshold = 0.50

        if phishing_probability >= threshold:

            label = 1
            classification = "phishing"

        else:

            label = 0
            classification = "legitimate"

        # ----------------------------------------------------
        # Return backend-friendly result
        # ----------------------------------------------------

        return {
            "label": label,
            "class": classification,

            "phishing_probability": round(
                phishing_probability,
                6
            ),

            "legitimate_probability": round(
                legitimate_probability,
                6
            ),

            "chunks_analyzed": len(
                chunks
            ),

            "threshold": threshold
        }

In [51]:
# ============================================================
# STEP 23: LOAD THE SAVED MODEL
# ============================================================

from inference import DistilBERTPhishingClassifier

classifier = DistilBERTPhishingClassifier(
    model_dir="./distilbert_model_final"
)

In [52]:
# ============================================================
# STEP 24: TEST PHISHING EMAIL
# ============================================================

phishing_email = """
URGENT SECURITY ALERT

Your account has been temporarily suspended due to
suspicious activity.

You must verify your account immediately to avoid permanent
closure.

Click the link below and enter your username, password,
and banking information.

Failure to verify your account within 24 hours will result
in permanent suspension.
"""

result = classifier.predict(phishing_email)

print(result)

{'label': 1, 'class': 'phishing', 'phishing_probability': 0.9976997971534729, 'legitimate_probability': 0.0023001530207693577, 'threshold': 0.5}


In [53]:
# ============================================================
# STEP 25: TEST LEGITIMATE EMAIL
# ============================================================

legitimate_email = """
Hi Ananya,

Just a reminder that our project meeting is scheduled
for tomorrow at 10 AM.

Please bring the updated presentation and the latest
testing results.

Thanks,
Project Team
"""

result = classifier.predict(legitimate_email)

print(result)

{'label': 0, 'class': 'legitimate', 'phishing_probability': 0.00011470039316918701, 'legitimate_probability': 0.9998853206634521, 'threshold': 0.5}


In [54]:
# ============================================================
# STEP 26: TEST LONG EMAIL HANDLING
# ============================================================

# Repeat legitimate text enough times to exceed 512 tokens
long_email = """
Hi team,

This is a routine project update.
Our meeting is scheduled for tomorrow.
Please review the documents before the meeting.
Thank you.
""" * 200

result = classifier.predict(long_email)

print(result)

{'label': 0, 'class': 'legitimate', 'phishing_probability': 0.00010417099838377908, 'legitimate_probability': 0.9998958110809326, 'threshold': 0.5}


In [55]:
# ============================================================
# STEP 27: VERIFY INFERENCE MODULE
# ============================================================

import inspect

print(inspect.getsource(classifier.predict))

    @torch.inference_mode()
    def predict(self, email_body: str):
        email_body = str(email_body or "").strip()

        inputs = self.tokenizer(
            email_body,
            return_tensors="pt",
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        outputs = self.model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)[0]

        phishing_probability = float(probabilities[1].cpu())
        legitimate_probability = float(probabilities[0].cpu())

        label = int(phishing_probability >= self.threshold)

        return {
            "label": label,
            "class": "phishing" if label else "legitimate",
            "phishing_probability": phishing_probability,
            "legitimate_probability": legitimate_probability,
            "threshold": self.threshold,
        }



In [56]:
print(classifier.predict(long_email))

{'label': 0, 'class': 'legitimate', 'phishing_probability': 0.00010417099838377908, 'legitimate_probability': 0.9998958110809326, 'threshold': 0.5}


In [63]:
# ============================================================
# STEP 31: WRITE THE FINAL INFERENCE MODULE
# ============================================================

%%writefile inference.py

import json
import os

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


class DistilBERTPhishingClassifier:
    """
    Backend-ready wrapper for the final trained DistilBERT model.

    Input:
        Complete email body as a string.

    Output:
        One email-level phishing prediction.

    Long emails:
        Automatically split into overlapping chunks.
    """

    def __init__(
        self,
        model_dir="./distilbert_model_final",
        max_length=512,
        chunk_size=480,
        overlap=50,
        threshold=0.50
    ):

        # ----------------------------------------------------
        # Configuration
        # ----------------------------------------------------

        self.model_dir = model_dir
        self.max_length = max_length
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.threshold = threshold

        # Make sure overlap is smaller than chunk size
        if overlap >= chunk_size:
            raise ValueError(
                "overlap must be smaller than chunk_size"
            )

        # ----------------------------------------------------
        # Select device
        # ----------------------------------------------------

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # ----------------------------------------------------
        # Load tokenizer
        # ----------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_dir
        )

        # We manually handle long sequences ourselves.
        # This prevents the tokenizer from warning when it
        # encounters an email longer than 512 tokens.
        self.tokenizer.model_max_length = 10**9

        # ----------------------------------------------------
        # Load FINAL trained DistilBERT
        # ----------------------------------------------------

        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_dir
        )

        self.model.to(self.device)
        self.model.eval()

        # ----------------------------------------------------
        # Load label mapping
        # ----------------------------------------------------

        mapping_path = os.path.join(
            self.model_dir,
            "label_mapping.json"
        )

        if os.path.exists(mapping_path):

            with open(mapping_path, "r") as file:
                self.label_mapping = json.load(file)

        else:

            self.label_mapping = {
                "0": "legitimate",
                "1": "phishing"
            }

        print("DistilBERT loaded successfully.")
        print("Device:", self.device)

    # ========================================================
    # CREATE EMAIL CHUNKS
    # ========================================================

    def _create_chunks(self, email_body):
        """
        Split the complete email into overlapping token chunks.

        Each chunk contains at most 480 actual email tokens.
        DistilBERT adds its special tokens afterwards, keeping
        the final sequence safely below the 512-token limit.
        """

        # Convert the complete email into token IDs.
        #
        # IMPORTANT:
        # add_special_tokens=False because we will add them
        # automatically when passing each chunk to the model.
        tokens = self.tokenizer.encode(
            email_body,
            add_special_tokens=False
        )

        # ----------------------------------------------------
        # Short email
        # ----------------------------------------------------

        if len(tokens) <= self.chunk_size:
            return [email_body]

        # ----------------------------------------------------
        # Long email
        # ----------------------------------------------------

        chunks = []

        step = self.chunk_size - self.overlap

        start = 0

        while start < len(tokens):

            # Take one section of the email
            chunk_tokens = tokens[
                start:start + self.chunk_size
            ]

            # Convert token IDs back into text
            chunk_text = self.tokenizer.decode(
                chunk_tokens,
                skip_special_tokens=True
            )

            chunks.append(chunk_text)

            # Move forward while retaining overlap
            start += step

        return chunks

    # ========================================================
    # PREDICT ONE CHUNK
    # ========================================================

    @torch.inference_mode()
    def _predict_chunk(self, chunk):
        """
        Run DistilBERT on one chunk.
        """

        # Tokenize ONE chunk.
        #
        # Because chunk_size = 480, the resulting sequence
        # including special tokens remains <= 512.
        inputs = self.tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length
        )

        # Move tensors to the same device as the model
        inputs = {
            key: value.to(self.device)
            for key, value in inputs.items()
        }

        # Run DistilBERT
        outputs = self.model(**inputs)

        # Convert logits into probabilities
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )[0]

        # Class 0 = legitimate
        # Class 1 = phishing

        legitimate_probability = float(
            probabilities[0].cpu()
        )

        phishing_probability = float(
            probabilities[1].cpu()
        )

        return {
            "legitimate_probability": legitimate_probability,
            "phishing_probability": phishing_probability
        }

    # ========================================================
    # PREDICT COMPLETE EMAIL
    # ========================================================

    def predict(self, email_body):
        """
        Analyze a complete email.

        Short email:
            One DistilBERT prediction.

        Long email:
            Multiple overlapping predictions are generated
            and combined into one email-level result.
        """

        # ----------------------------------------------------
        # Validate input
        # ----------------------------------------------------

        if email_body is None:
            raise ValueError(
                "email_body cannot be None"
            )

        email_body = str(email_body).strip()

        if not email_body:
            raise ValueError(
                "email_body cannot be empty"
            )

        # ----------------------------------------------------
        # Create chunks
        # ----------------------------------------------------

        chunks = self._create_chunks(
            email_body
        )

        # ----------------------------------------------------
        # Predict every chunk
        # ----------------------------------------------------

        chunk_results = []

        for chunk in chunks:

            result = self._predict_chunk(
                chunk
            )

            chunk_results.append(result)

        # ----------------------------------------------------
        # Aggregate predictions
        # ----------------------------------------------------
        #
        # MAX is used deliberately.
        #
        # If one part of a long email is strongly suspicious,
        # that signal should not be diluted by harmless sections.
        # ----------------------------------------------------

        phishing_probability = max(
            result["phishing_probability"]
            for result in chunk_results
        )

        legitimate_probability = (
            1.0 - phishing_probability
        )

        # ----------------------------------------------------
        # Final classification
        # ----------------------------------------------------

        label = int(
            phishing_probability >= self.threshold
        )

        classification = (
            "phishing"
            if label == 1
            else "legitimate"
        )

        # ----------------------------------------------------
        # Return backend-friendly JSON-compatible result
        # ----------------------------------------------------

        return {
            "label": label,
            "class": classification,
            "phishing_probability": round(
                phishing_probability,
                6
            ),
            "legitimate_probability": round(
                legitimate_probability,
                6
            ),
            "chunks_analyzed": len(chunks),
            "threshold": self.threshold
        }

Overwriting inference.py


In [64]:
# ============================================================
# STEP 28: RELOAD THE UPDATED INFERENCE MODULE
# ============================================================

import importlib
import inference

importlib.reload(inference)

from inference import DistilBERTPhishingClassifier

classifier = DistilBERTPhishingClassifier(
    model_dir="./distilbert_model_final"
)

DistilBERT loaded successfully.
Device: cuda


In [65]:
# ============================================================
# STEP 29: VERIFY CHUNKING IS REALLY ACTIVE
# ============================================================

import inspect

print(
    inspect.getsource(
        DistilBERTPhishingClassifier.predict
    )
)

    def predict(self, email_body):
        """
        Analyze a complete email.

        Short email:
            One DistilBERT prediction.

        Long email:
            Multiple overlapping predictions are generated
            and combined into one email-level result.
        """

        # ----------------------------------------------------
        # Validate input
        # ----------------------------------------------------

        if email_body is None:
            raise ValueError(
                "email_body cannot be None"
            )

        email_body = str(email_body).strip()

        if not email_body:
            raise ValueError(
                "email_body cannot be empty"
            )

        # ----------------------------------------------------
        # Create chunks
        # ----------------------------------------------------

        chunks = self._create_chunks(
            email_body
        )

        # -------------------------------------------

In [66]:
# ============================================================
# STEP 30: TEST LONG EMAIL
# ============================================================

result = classifier.predict(long_email)

print(result)

{'label': 0, 'class': 'legitimate', 'phishing_probability': 0.000177, 'legitimate_probability': 0.999823, 'chunks_analyzed': 14, 'threshold': 0.5}


In [67]:
# ============================================================
# STEP 34: VERIFY EVERY CHUNK IS WITHIN DISTILBERT'S LIMIT
# ============================================================

chunks = classifier._create_chunks(long_email)

chunk_lengths = []

for chunk in chunks:

    token_count = len(
        classifier.tokenizer(
            chunk,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )

    chunk_lengths.append(token_count)

print("Total chunks:", len(chunks))
print("Maximum chunk length:", max(chunk_lengths))
print("Minimum chunk length:", min(chunk_lengths))

print(
    "All chunks <= 512:",
    all(length <= 512 for length in chunk_lengths)
)

print("\nChunk lengths:")
print(chunk_lengths)

Total chunks: 14
Maximum chunk length: 482
Minimum chunk length: 12
All chunks <= 512: True

Chunk lengths:
[482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 482, 442, 12]


In [68]:
# ============================================================
# STEP 35: CREATE BACKEND-READY DISTILBERT PACKAGE
# ============================================================

import os
import shutil

# Final package directory
PACKAGE_DIR = "distilbert_phishing_backend"

# Remove an old package if one exists
if os.path.exists(PACKAGE_DIR):
    shutil.rmtree(PACKAGE_DIR)

# Create directories
MODEL_DIR = os.path.join(
    PACKAGE_DIR,
    "model"
)

os.makedirs(MODEL_DIR)

# ------------------------------------------------------------
# Copy the trained model files
# ------------------------------------------------------------

SOURCE_MODEL_DIR = "distilbert_model_final"

for filename in os.listdir(SOURCE_MODEL_DIR):

    source = os.path.join(
        SOURCE_MODEL_DIR,
        filename
    )

    destination = os.path.join(
        MODEL_DIR,
        filename
    )

    if os.path.isfile(source):
        shutil.copy2(
            source,
            destination
        )

# ------------------------------------------------------------
# Copy inference.py
# ------------------------------------------------------------

shutil.copy2(
    "inference.py",
    os.path.join(
        PACKAGE_DIR,
        "inference.py"
    )
)

print("Package created successfully!")
print("\nPackage contents:")

for root, dirs, files in os.walk(PACKAGE_DIR):

    level = root.replace(
        PACKAGE_DIR,
        ""
    ).count(os.sep)

    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

Package created successfully!

Package contents:
distilbert_phishing_backend/
    inference.py
    model/
        tokenizer_config.json
        training_args.bin
        tokenizer.json
        config.json
        label_mapping.json
        vocab.txt
        special_tokens_map.json
        model.safetensors


In [70]:
# ============================================================
# STEP 36: CREATE README SAFELY
# ============================================================

import os

readme_lines = [
    "# CyberMail Intelligence - DistilBERT Phishing Classifier",
    "",
    "This package contains the final fine-tuned DistilBERT model",
    "used to classify email bodies as legitimate or phishing.",
    "",
    "## Model",
    "",
    "Base model: distilbert-base-uncased",
    "Task: Binary phishing-email classification",
    "Maximum model sequence length: 512 tokens",
    "",
    "## Performance",
    "",
    "Internal held-out test:",
    "- Accuracy: 99.32%",
    "- Precision: 99.38%",
    "- Recall: 98.78%",
    "- F1: 99.08%",
    "- ROC-AUC: 99.95%",
    "",
    "External test dataset:",
    "- Accuracy: 99.01%",
    "- Precision: 99.39%",
    "- Recall: 98.61%",
    "- F1: 99.00%",
    "- ROC-AUC: 99.94%",
    "",
    "## Usage",
    "",
    "```python",
    "from inference import DistilBERTPhishingClassifier",
    "",
    'classifier = DistilBERTPhishingClassifier(model_dir="./model")',
    "",
    "result = classifier.predict(email_body)",
    "print(result)",
    "```",
    "",
    "## Long Emails",
    "",
    "Emails longer than the DistilBERT sequence limit are",
    "automatically divided into overlapping chunks.",
    "",
    "Each chunk is independently evaluated.",
    "",
    "The highest phishing probability is used as the",
    "email-level phishing probability.",
    "",
    "## Backend Integration",
    "",
    "The backend should pass the complete parsed email body",
    "to classifier.predict(email_body).",
    "",
    "The classifier returns one JSON-compatible result.",
    "",
    "The DistilBERT model does not parse .eml files itself.",
    "The email parser should extract the email body before",
    "calling the classifier.",
]

readme_path = os.path.join(
    PACKAGE_DIR,
    "README.md"
)

with open(readme_path, "w", encoding="utf-8") as file:
    file.write("\n".join(readme_lines))

print("README created successfully.")
print("Location:", readme_path)

README created successfully.
Location: distilbert_phishing_backend/README.md


In [71]:
# ============================================================
# STEP 37: VERIFY FINAL PACKAGE
# ============================================================

required_files = [
    "model/config.json",
    "model/model.safetensors",
    "model/tokenizer.json",
    "model/tokenizer_config.json",
    "model/special_tokens_map.json",
    "model/vocab.txt",
    "model/label_mapping.json",
    "inference.py",
    "README.md"
]

print("FINAL PACKAGE CHECK")
print("===================")

all_present = True

for file in required_files:

    path = os.path.join(
        PACKAGE_DIR,
        file
    )

    exists = os.path.isfile(path)

    print(
        f"{'✓' if exists else '✗'} {file}"
    )

    if not exists:
        all_present = False

print("\nPackage ready:", all_present)

FINAL PACKAGE CHECK
✓ model/config.json
✓ model/model.safetensors
✓ model/tokenizer.json
✓ model/tokenizer_config.json
✓ model/special_tokens_map.json
✓ model/vocab.txt
✓ model/label_mapping.json
✓ inference.py
✓ README.md

Package ready: True


In [72]:
# ============================================================
# STEP 38: CREATE FINAL DISTILBERT ZIP
# ============================================================

import os
import shutil

PACKAGE_DIR = "distilbert_phishing_backend"
ZIP_NAME = "distilbert_phishing_backend"

# Create ZIP
zip_path = shutil.make_archive(
    ZIP_NAME,
    "zip",
    PACKAGE_DIR
)

print("ZIP CREATED SUCCESSFULLY")
print("========================")
print("File:", zip_path)

# Check file size
size_mb = os.path.getsize(zip_path) / (1024 * 1024)

print(f"Size: {size_mb:.2f} MB")

ZIP CREATED SUCCESSFULLY
File: /content/distilbert_phishing_final/distilbert_phishing_backend.zip
Size: 235.86 MB


In [74]:
# ============================================================
# STEP 39: DOWNLOAD FINAL DISTILBERT PACKAGE
# ============================================================

from google.colab import files

files.download(
    "/content/distilbert_phishing_final/distilbert_phishing_backend.zip   "
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>